In [2]:
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# 1. Load Data
fake = pd.read_csv('Fake.csv')
true = pd.read_csv('True.csv')

# 2. Data Preprocessing
fake['label'] = 0
true['label'] = 1
combined = pd.concat([fake, true], axis=0)
combined.dropna(inplace=True)
combined['content'] = combined['title'] + ' ' + combined['text']

# 3. Train-Test Split
X = combined['content']
y = combined['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Text Vectorization and Feature Selection
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=2000, ngram_range=(1, 2))),  # Reduced to 2000 features
    ('select', SelectKBest(chi2, k=1000))  # Select top 1000 features
])

X_train_transformed = pipeline.fit_transform(X_train, y_train)
X_test_transformed = pipeline.transform(X_test)

# 5. Model Training
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Naive Bayes': MultinomialNB(),
    'Random Forest': RandomForestClassifier(n_estimators=100)
}

predictions = []
for name, model in models.items():
    model.fit(X_train_transformed, y_train)
    y_pred = model.predict(X_test_transformed)
    predictions.append(y_pred)
    
    print(f"\n{name} Results:")
    print(classification_report(y_test, y_pred))
    print(f"Accuracy: {accuracy_score(y_test, y_pred)}")

# 6. Ensemble Prediction and Accuracy
ensemble_preds = np.mean(predictions, axis=0)
ensemble_preds_class = (ensemble_preds > 0.5).astype(int)  # Threshold at 0.5 for classification
ensemble_accuracy = accuracy_score(y_test, ensemble_preds_class)

print("\nEnsemble Model Results:")
print(classification_report(y_test, ensemble_preds_class))
print(f"Ensemble Accuracy: {ensemble_accuracy:.4f}")

# 7. SHAP Explainability
background = X_train_transformed[:100].toarray()  # Convert to dense
test_sample = X_test_transformed[:20].toarray()  # Limit test samples

shap_values_ensemble = np.zeros(test_sample.shape)

for name, model in models.items():
    if isinstance(model, MultinomialNB):
        # Use KernelExplainer for Naive Bayes
        explainer = shap.KernelExplainer(model.predict_proba, background)
        shap_values = explainer.shap_values(test_sample, nsamples=100)
        shap_values_model = np.array(shap_values[1])  # Class 1 (positive class)
    else:
        # PermutationExplainer for Logistic Regression and Random Forest
        explainer = shap.PermutationExplainer(model.predict, background)
        shap_values = explainer(test_sample, max_evals=7500)  # Adjust max_evals to required level
        shap_values_model = shap_values.values if shap_values.values.ndim == 2 else shap_values.values[:, :, 1]
    
    # Aggregate SHAP values for ensemble
    shap_values_ensemble += shap_values_model

# Average SHAP values across models
shap_values_ensemble /= len(models)

# 8. Visualize SHAP Results
plt.title("SHAP Summary Plot - Ensemble Model")
shap.summary_plot(shap_values_ensemble, test_sample)



Logistic Regression Results:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      4733
           1       0.98      0.99      0.99      4247

    accuracy                           0.99      8980
   macro avg       0.99      0.99      0.99      8980
weighted avg       0.99      0.99      0.99      8980

Accuracy: 0.9861915367483296

Naive Bayes Results:
              precision    recall  f1-score   support

           0       0.95      0.95      0.95      4733
           1       0.94      0.95      0.95      4247

    accuracy                           0.95      8980
   macro avg       0.95      0.95      0.95      8980
weighted avg       0.95      0.95      0.95      8980

Accuracy: 0.9482182628062361

Random Forest Results:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      4733
           1       1.00      1.00      1.00      4247

    accuracy                           1.00     

PermutationExplainer explainer: 21it [02:15,  6.80s/it]                                                                


  0%|          | 0/20 [00:00<?, ?it/s]

D:\Downloads\cuda\lib\site-packages\sklearn\linear_model\_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 24 iterations, alpha=5.494e-02, previous alpha=5.492e-02, with an active set of 13 regressors.
  warnings.warn(


ValueError: You are using LassoLarsIC in the case where the number of samples is smaller than the number of features. In this setting, getting a good estimate for the variance of the noise is not possible. Provide an estimate of the noise variance in the constructor.

In [4]:
# Import necessary libraries for BERT
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import (
    Trainer, 
    TrainingArguments, 
    BertTokenizer, 
    BertForSequenceClassification,
    DataCollatorWithPadding
)
from torch.utils.data import Dataset
import torch
import os

# Define a main function to ensure compatibility, especially on Windows
def main():
    # Step 1: Load and preprocess the dataset
    # Load datasets
    fake_df = pd.read_csv('Fake.csv')
    true_df = pd.read_csv('True.csv')
    
    # Check if necessary columns exist
    required_columns = {'title', 'text'}
    if not required_columns.issubset(fake_df.columns) or not required_columns.issubset(true_df.columns):
        raise ValueError(f"Input CSV files must contain the columns: {required_columns}")
    
    # Add labels
    fake_df['label'] = 0
    true_df['label'] = 1
    
    # Combine and shuffle
    combined_df = pd.concat([fake_df, true_df], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)
    
    # Create a content column
    combined_df['content'] = combined_df['title'].astype(str) + " " + combined_df['text'].astype(str)
    
    # Drop missing values
    combined_df = combined_df.dropna(subset=['content', 'label'])
    
    # Split data
    X = combined_df['content']
    y = combined_df['label']
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    
    # Step 2: Custom Dataset class
    class NewsDataset(Dataset):
        def __init__(self, texts, labels, tokenizer, max_length=512):
            self.encodings = tokenizer(
                texts, 
                truncation=True, 
                padding=True, 
                max_length=max_length
            )
            self.labels = labels.tolist()
    
        def __getitem__(self, idx):
            item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
            return item
    
        def __len__(self):
            return len(self.labels)
    
    train_dataset = NewsDataset(X_train.tolist(), y_train, tokenizer)
    test_dataset = NewsDataset(X_test.tolist(), y_test, tokenizer)
    
    # Load the BERT model for sequence classification
    model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
    
    # Training arguments for the BERT model
    training_args = TrainingArguments(
        output_dir='./results',
        num_train_epochs=3,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        warmup_steps=500,
        weight_decay=0.01,
        logging_dir='./logs',
        logging_steps=100,  # Reduced frequency to avoid clutter
        eval_strategy="epoch",       # Updated parameter name
        save_strategy="epoch",       # Ensure save_strategy matches eval_strategy
        fp16=torch.cuda.is_available(),  # Enable mixed precision only if GPU is available
        gradient_accumulation_steps=4,
        dataloader_num_workers=2,    # Reduced to avoid potential issues
        load_best_model_at_end=True, # Useful for evaluation
        metric_for_best_model='accuracy',  # Define a metric for selecting the best model
    )
    
    # Define a simple accuracy metric
    from sklearn.metrics import accuracy_score, precision_recall_fscore_support
    
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = torch.argmax(torch.tensor(logits), dim=-1)
        acc = accuracy_score(labels, predictions)
        precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')
        return {
            'accuracy': acc,
            'precision': precision,
            'recall': recall,
            'f1': f1,
        }
    
    # Trainer initialization
    data_collator = DataCollatorWithPadding(tokenizer)
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        data_collator=data_collator,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics
    )
    
    # Train the model
    try:
        trainer.train()
    except Exception as e:
        print(f"Training failed: {e}")
        return  # Exit if training fails
    
    # Evaluate the model
    results = trainer.evaluate()
    print("Evaluation Results:", results)
    
    # Step 9: Save the fine-tuned model
    save_directory = './fine_tuned_bert'
    os.makedirs(save_directory, exist_ok=True)
    trainer.save_model(save_directory)
    tokenizer.save_pretrained(save_directory)
    
    # Step 10: Load the fine-tuned model for inference
    fine_tuned_model = BertForSequenceClassification.from_pretrained(save_directory)
    fine_tuned_tokenizer = BertTokenizer.from_pretrained(save_directory)
    
    # Move model to the appropriate device
    device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
    fine_tuned_model.to(device)
    
    # Example inference
    test_sentence = "This is a test sentence."
    inputs = fine_tuned_tokenizer(
        test_sentence, 
        return_tensors="pt", 
        truncation=True, 
        padding=True, 
        max_length=512
    )
    inputs = {key: val.to(device) for key, val in inputs.items()}  # Move inputs to device
    with torch.no_grad():
        outputs = fine_tuned_model(**inputs)
    predictions = torch.argmax(outputs.logits, dim=-1)
    print("Predicted class:", predictions.item())

# Ensure the main function is called
if __name__ == "__main__":
    main()


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\Sharan\AppData\Local\Temp\ipykernel_13820\3396378790.py:111: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Training failed: Can't pickle local object 'main.<locals>.NewsDataset'
